# ABL1 4-Way Controlled Experiment

Tests whether SPADE's PAE-guided rigid-body domain rearrangement adds discriminative signal
beyond NMA-based ensembles and Vina's own stochastic noise.

## Experimental design

| Condition | Receptor | Key variable |
|---|---|---|
| **A** | PAE-weighted NMA conformers | seed = BASE_SEED + i; max Cα disp ~0.4Å |
| **B** | Uniform NMA conformers | seed = BASE_SEED + i; same disp, no PAE bias |
| **C** | Raw AlphaFold structure × 10 runs | seed = BASE_SEED + i; pure Vina stochasticity |
| **D** | PAE domain rearrangement conformers | seed = BASE_SEED + i; Phe382 disp up to ~5Å |

Matched seeds across all four conditions isolate conformational signal from random-walk variance.

## Ligands
- **Dasatinib** — Type I inhibitor, binds DFG-in ABL1 (should converge in all conditions)
- **Imatinib** — Type II inhibitor, requires DFG-out pocket (key test: fragments only in D?)

## Calibration
- ABL1 DFG-in (2GQG) vs DFG-out (2HYY): activation loop RMSD = 16.94Å, Phe382 disp = 6.81Å
- AlphaFold PAE (activation loop vs N-lobe) = 9.53Å
- Scale factor = 1.85 (PAE underestimates actual displacement by ~0.54×)
- Rotation direction: +1 (CCW), confirmed consistent across Asp381–Gly388
- Hinge auto-detected: Cα(380) → Cα(402) for ABL1

## Decision rule (pre-registered)
If imatinib clusters >> dasatinib clusters **only in condition D**, PAE domain rearrangement
is the discriminating mechanism. If A~B~C also show the same pattern, the signal is geometric
(ligand frustration in a rigid pocket). If no condition discriminates, the story fails.

## 1. Setup

In [ ]:
import subprocess, sys, shutil

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "rdkit", "prody", "prolif", "numba", "meeko", "gemmi",
    "pandas", "plotly", "pdb2pqr"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "git+https://github.com/ChandraguptSharma07/Spade.git"], check=False)

_UNIDOCK_PATH = "/usr/local/bin/unidock"
_UNIDOCK_RELEASES = [
    "https://github.com/dptech-corp/Uni-Dock/releases/download/1.1.0/unidock-1.1.0-cuda122-linux-x86_64",
    "https://github.com/dptech-corp/Uni-Dock/releases/download/1.1.0/unidock-1.1.0-cuda120-linux-x86_64",
]

def _install_unidock():
    if shutil.which("unidock"):
        return True
    for url in _UNIDOCK_RELEASES:
        r = subprocess.run(["wget", "-q", url, "-O", _UNIDOCK_PATH], capture_output=True)
        if r.returncode != 0:
            continue
        subprocess.run(["chmod", "+x", _UNIDOCK_PATH], check=True)
        probe = subprocess.run([_UNIDOCK_PATH, "--help"], capture_output=True, timeout=10)
        if b"UniDock" in probe.stdout + probe.stderr or probe.returncode in (0, 1):
            print(f"UniDock installed: {url.split('/')[-1]}")
            return True
        import os; os.unlink(_UNIDOCK_PATH)
    return False

_unidock_available = _install_unidock()
if not _unidock_available:
    print("WARNING: UniDock unavailable — will use CPU Vina")

def _detect_backend():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                            "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
        if r.returncode == 0 and r.stdout.strip():
            print("GPU:", r.stdout.strip())
            if _unidock_available:
                print("Backend: gpu (UniDock)")
                return "gpu"
    except Exception:
        pass
    print("Backend: cpu (Vina)")
    return "cpu"

BACKEND = _detect_backend()

N_CONFORMERS    = 10
BASE_SEED       = 1
N_POSES         = 9
GPU_SEARCH_MODE = "fast"

# Domain rearrangement calibration (ABL1 2GQG/2HYY)
PAE_SCALE_FACTOR   = 1.85
ROTATION_DIRECTION = 1   # +1 = CCW = DFG-out direction for ABL1

print(f"N_CONFORMERS={N_CONFORMERS}, BASE_SEED={BASE_SEED}, BACKEND={BACKEND!r}")

## 2. Fetch ABL1 and define pocket

In [ ]:
import numpy as np
from spade.core.structure import fetch_structure

print("Fetching ABL1 (P00519)...")
abl1 = fetch_structure("P00519")
print(f"  {abl1.n_residues} residues, AF version {abl1.af_version}")

ca_sel    = abl1.atoms.select("calpha")
ca_coords = ca_sel.getCoords()
resnums   = ca_sel.getResnums()

POCKET_RESNUMS = (list(range(248, 257)) + list(range(315, 322))
                  + list(range(354, 363)) + list(range(381, 403)))
POCKET_INDICES = np.array([
    np.where(resnums == r)[0][0]
    for r in POCKET_RESNUMS if len(np.where(resnums == r)[0]) > 0
])
print(f"  Pocket: {len(POCKET_INDICES)} residues")

print("\n  DFG loop (D381, F382, G383):")
for r in [381, 382, 383]:
    idx = np.where(resnums == r)[0][0]
    region = list(range(max(0, idx-10), min(abl1.n_residues, idx+11)))
    pae_local = abl1.pae_matrix[idx, region].mean()
    print(f"    Res {r}: pLDDT={abl1.plddt[idx]:.1f}  PAE_local={pae_local:.1f}A")

## 3. Prepare ligands

In [ ]:
from spade.core.ligand import prepare_ligand
from rdkit.Chem import rdMolDescriptors

LIGANDS = {
    "Imatinib":  "Cc1ccc(Nc2nccc(-c3cccnc3)n2)cc1NC(=O)c1ccc(CN2CCN(C)CC2)cc1",
    "Dasatinib": "Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCO)CC2)n1",
}

prepared = {}
for name, smiles in LIGANDS.items():
    variants = prepare_ligand(smiles, ph=7.4, enumerate_stereo=False)
    if not variants:
        raise RuntimeError(f"{name} preparation failed")
    prepared[name] = variants[0]
    mol = prepared[name].mol
    print(f"  {name}: {mol.GetNumHeavyAtoms()} heavy atoms, "
          f"{rdMolDescriptors.CalcNumRotatableBonds(mol)} rot. bonds")

LIGAND_NAMES = list(prepared.keys())
LIGAND_LIST  = [prepared[n] for n in LIGAND_NAMES]

## 4. Generate conformers — conditions A, B, C (NMA-based)

In [ ]:
from spade.core.flexibility import build_flexibility_profile, FlexibilityProfile
from spade.core.ensemble import EnsembleGenerator

# Condition A: PAE-weighted NMA
print("Building PAE-weighted flexibility profile (Condition A)...")
pae_profile = build_flexibility_profile(
    plddt=abl1.plddt, pae_matrix=abl1.pae_matrix,
    pocket_residues=POCKET_INDICES, ca_coords=ca_coords,
)
np.random.seed(42)
conformers_A = EnsembleGenerator(abl1, pae_profile, n_conformers=N_CONFORMERS).generate()
print(f"  {len(conformers_A)} conformers generated")

# Condition B: Uniform NMA
print("Building uniform flexibility profile (Condition B)...")
uniform_profile = FlexibilityProfile(
    residue_tiers=pae_profile.residue_tiers,
    flexibility_graph=pae_profile.flexibility_graph,
    pocket_residues=POCKET_INDICES,
    disordered_residues=pae_profile.disordered_residues,
    mode_weight_vector=np.ones(abl1.n_residues, dtype=np.float32),
    inter_domain_pae_warning=pae_profile.inter_domain_pae_warning,
)
np.random.seed(42)
conformers_B = EnsembleGenerator(abl1, uniform_profile, n_conformers=N_CONFORMERS).generate()
print(f"  {len(conformers_B)} conformers generated")

# Condition C: raw AlphaFold structure (seed-only variation)
print(f"Condition C: raw AF structure x{N_CONFORMERS}")
conformers_C = [abl1.atoms] * N_CONFORMERS

# Verify NMA displacement is sub-Angstrom (expected ~0.4A max)
ref_ca = abl1.atoms.select("calpha")
ref_rn = ref_ca.getResnums()
for cond_name, confs in [("A", conformers_A), ("B", conformers_B)]:
    disps = []
    for conf in confs:
        ca = conf.select("calpha")
        rn = ca.getResnums()
        for r in [381, 382, 383]:
            i_r = np.where(ref_rn == r)[0]; i_c = np.where(rn == r)[0]
            if len(i_r) > 0 and len(i_c) > 0:
                disps.append(np.linalg.norm(ref_ca.getCoords()[i_r[0]] - ca.getCoords()[i_c[0]]))
    print(f"  Cond {cond_name} DFG Calpha: mean={np.mean(disps):.3f}A  max={np.max(disps):.3f}A")

## 5. Generate conformers — condition D (PAE domain rearrangement)

Rotates the activation loop (auto-detected) as a rigid body around the hinge axis.
10 conformers span DFG-in (angle=0) to DFG-out (angle=theta_max) in equal steps.

In [ ]:
import warnings
from spade.core.domain_rearrangement import (
    identify_mobile_segment, PAEDomainRearrangementGenerator
)

print("Identifying mobile segment from inter-domain PAE...")
segment = identify_mobile_segment(
    abl1,
    reference_domain_resnums=list(range(235, 311)),  # N-lobe
    min_inter_pae=6.0,
    min_segment_length=5,
    max_segment_length=60,
    min_plddt=50.0,
    search_resnums=list(range(230, 510)),             # kinase domain only
)

if segment is None:
    raise RuntimeError("No mobile segment found — check PAE thresholds")

seg_resnums = [int(resnums[i]) for i in segment.residues if i < len(resnums)]
print(f"  Mobile segment: res {seg_resnums[0]}-{seg_resnums[-1]} ({len(seg_resnums)} residues)")
print(f"  Hinge: Cα({segment.hinge_n_resnum}) → Cα({segment.hinge_c_resnum})")
print(f"  Mean inter-domain PAE: {segment.mean_inter_pae:.2f} A")

gen_D = PAEDomainRearrangementGenerator(
    structure=abl1,
    mobile_segment=segment,
    inter_domain_pae=segment.mean_inter_pae,
    pae_scale_factor=PAE_SCALE_FACTOR,
    n_conformers=N_CONFORMERS,
    rotation_direction=ROTATION_DIRECTION,
)
print(f"  theta_max = {gen_D.theta_max_deg:.1f} deg")
print(f"  (Crystal DFG-in→out Phe382 disp = 6.81A; lever arm calibrated to match)")

print(f"\nGenerating {N_CONFORMERS} domain rearrangement conformers (Condition D)...")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    conformers_D = gen_D.generate()
for w in caught:
    print(f"  WARN: {w.message}")
print(f"  {len(conformers_D)} conformers generated")

# Verify Phe382 displacement profile
idx382 = np.where(ref_rn == 382)[0]
ref_phe = ref_ca.getCoords()[idx382[0]]
print("\n  Phe382 Ca displacement (target: 6.81A at full rotation):")
for i, conf in enumerate(conformers_D):
    angle = float(conf.getData("rotation_angle_deg")[0])
    ca = conf.select("calpha")
    rn = ca.getResnums()
    idx = np.where(rn == 382)[0]
    disp = np.linalg.norm(ca.getCoords()[idx[0]] - ref_phe) if len(idx) > 0 else 0.0
    bar = "#" * int(disp)
    print(f"    Conf {i:2d} (angle={angle:5.1f} deg): {disp:.2f} A  {bar}")

## 6. Docking — all four conditions with matched seeds

Conformer `i` uses `seed = BASE_SEED + i` across all conditions.
Any cluster count difference is attributable to receptor geometry, not random-walk variance.

In [ ]:
from spade.core.docking import compute_bounding_box, get_docking_engine, DockingResult
import time

def run_condition(label, conformers, ligand_list, ligand_names,
                  pocket_indices, base_seed, backend, n_poses=9):
    print(f"\n=== Condition {label}: {len(conformers)} conformers ===")
    all_results = {name: [] for name in ligand_names}
    t0 = time.perf_counter()

    for conf_idx, conformer in enumerate(conformers):
        seed = base_seed + conf_idx
        bbox = compute_bounding_box(conformer, pocket_indices)
        engine = get_docking_engine(
            backend=backend, exhaustiveness=8, scoring="vina",
            search_mode=GPU_SEARCH_MODE if backend == "gpu" else None,
        )
        import warnings
        with warnings.catch_warnings(record=True):
            warnings.simplefilter("always")
            if backend == "gpu":
                batch_poses = engine.dock_batch(
                    conformer, ligand_list, bbox, n_poses, conf_idx, seed=seed
                )
                for name, poses in zip(ligand_names, batch_poses):
                    rmsd_data = conformer.getData("ca_rmsd_from_ref")
                    ca_rmsd = float(rmsd_data[0]) if rmsd_data is not None else 0.0
                    all_results[name].append(DockingResult(
                        conformer_index=conf_idx, conformer_ca_rmsd=ca_rmsd,
                        poses=poses, bounding_box=bbox, docking_time_seconds=0.0,
                    ))
            else:
                for name, lig in zip(ligand_names, ligand_list):
                    poses = engine.dock(
                        conformer, lig, bbox, n_poses, conf_idx, seed=seed
                    )
                    rmsd_data = conformer.getData("ca_rmsd_from_ref")
                    ca_rmsd = float(rmsd_data[0]) if rmsd_data is not None else 0.0
                    all_results[name].append(DockingResult(
                        conformer_index=conf_idx, conformer_ca_rmsd=ca_rmsd,
                        poses=poses, bounding_box=bbox, docking_time_seconds=0.0,
                    ))

        best = {n: (f"{all_results[n][-1].poses[0].score_kcal_mol:.2f}"
                    if all_results[n][-1].poses else "N/A") for n in ligand_names}
        print(f"  [{conf_idx+1}/{len(conformers)}] seed={seed}  "
              + "  ".join(f"{n}: {s}" for n, s in best.items()))

    print(f"  Done in {time.perf_counter()-t0:.1f}s")
    return all_results


results_A = run_condition("A (PAE NMA)",      conformers_A, LIGAND_LIST, LIGAND_NAMES,
                          POCKET_INDICES, BASE_SEED, BACKEND)
results_B = run_condition("B (Uniform NMA)",  conformers_B, LIGAND_LIST, LIGAND_NAMES,
                          POCKET_INDICES, BASE_SEED, BACKEND)
results_C = run_condition("C (Single struct)",conformers_C, LIGAND_LIST, LIGAND_NAMES,
                          POCKET_INDICES, BASE_SEED, BACKEND)
results_D = run_condition("D (Domain rearg)", conformers_D, LIGAND_LIST, LIGAND_NAMES,
                          POCKET_INDICES, BASE_SEED, BACKEND)

## 7. Cluster poses and compute fragmentation metrics

In [ ]:
from spade.core.clustering import cluster_poses
import pandas as pd

def cluster_condition(label, results_dict, conformers, ligand_names, prepared_dict):
    rows = []
    for name in ligand_names:
        mol = prepared_dict[name].mol
        try:
            consensus = cluster_poses(results_dict[name], conformers, mol)
            tc = consensus.top_cluster
            best_score = tc.representative_pose.score_kcal_mol if tc else 0.0
            rows.append({
                "Condition":       label,
                "Ligand":          name,
                "N Clusters":      consensus.n_clusters,
                "Best Score":      round(best_score, 3),
                "Coverage":        round(tc.fraction_ensemble if tc else 0.0, 3),
                "Consensus Score": round(tc.consensus_score if tc else 0.0, 3),
                "FP Method":       consensus.fingerprint_method,
            })
        except Exception as e:
            print(f"  Clustering failed {name} ({label}): {e}")
            rows.append({"Condition": label, "Ligand": name,
                         "N Clusters": -1, "Best Score": 0.0,
                         "Coverage": 0.0, "Consensus Score": 0.0, "FP Method": "error"})
    return rows

rows = []
rows += cluster_condition("A: PAE NMA",      results_A, conformers_A, LIGAND_NAMES, prepared)
rows += cluster_condition("B: Uniform NMA",  results_B, conformers_B, LIGAND_NAMES, prepared)
rows += cluster_condition("C: Single struct",results_C, conformers_C, LIGAND_NAMES, prepared)
rows += cluster_condition("D: Domain rearg", results_D, conformers_D, LIGAND_NAMES, prepared)

df = pd.DataFrame(rows)
df.to_csv("abl1_4way_results.csv", index=False)
print(df.to_string())

## 8. Key comparison table

In [ ]:
conditions = ["A: PAE NMA", "B: Uniform NMA", "C: Single struct", "D: Domain rearg"]

pivot_clusters = df.pivot(index="Ligand", columns="Condition", values="N Clusters")
pivot_scores   = df.pivot(index="Ligand", columns="Condition", values="Best Score")

print("=" * 70)
print("CLUSTER COUNT (higher = more fragmentation = worse fit)")
print("=" * 70)
print(pivot_clusters[conditions].to_string())
print()
print("BEST DOCKING SCORE (kcal/mol)")
print(pivot_scores[conditions].to_string())
print()

print("=" * 70)
print("INTERPRETATION")
print("=" * 70)
for ligand in LIGAND_NAMES:
    if ligand not in pivot_clusters.index:
        continue
    row = pivot_clusters.loc[ligand]
    a = row.get("A: PAE NMA"); b = row.get("B: Uniform NMA")
    c = row.get("C: Single struct"); d = row.get("D: Domain rearg")
    print(f"\n{ligand}: A={a}  B={b}  C={c}  D={d}")

print()
print("Decision rules:")
print("  D >> A,B,C for Imatinib only  -> Domain rearrangement is the discriminating mechanism")
print("  A~B~C~D for both ligands      -> No conformational signal; pure ligand geometry")
print("  Imatinib ~ Dasatinib in all   -> No Type II discrimination; story fails")

## 9. Visualisation

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = {"Imatinib": "#ff6b6b", "Dasatinib": "#4ecdc4"}
cond_labels = [c.split(":")[0] for c in conditions]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["N Clusters (fragmentation)", "Best Score (kcal/mol)"])

for ligand in LIGAND_NAMES:
    sub = df[df["Ligand"] == ligand]
    fig.add_trace(go.Bar(
        name=ligand,
        x=cond_labels,
        y=[sub[sub["Condition"]==c]["N Clusters"].values[0]
           if len(sub[sub["Condition"]==c]) > 0 else 0 for c in conditions],
        marker_color=colors.get(ligand, "gray"),
        legendgroup=ligand,
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        name=ligand, x=cond_labels,
        y=[sub[sub["Condition"]==c]["Best Score"].values[0]
           if len(sub[sub["Condition"]==c]) > 0 else 0 for c in conditions],
        marker_color=colors.get(ligand, "gray"),
        legendgroup=ligand, showlegend=False,
    ), row=1, col=2)

fig.update_layout(
    title="ABL1 4-Way Experiment: Imatinib (Type II) vs Dasatinib (Type I)",
    template="plotly_dark", barmode="group", height=500,
    annotations=[dict(
        text="D = PAE domain rearrangement (Phe382 up to ~5A displacement)",
        xref="paper", yref="paper", x=0.5, y=-0.15,
        showarrow=False, font=dict(color="gray", size=11)
    )]
)
fig.write_html("abl1_4way_results.html")
fig.show()

## 10. Interpretation guide

| Pattern | Conclusion |
|---|---|
| Imatinib clusters >> Dasatinib **only in D** | Domain rearrangement is the mechanism; NMA too small |
| Imatinib >> Dasatinib **in A, B, C, D** | Ligand frustration in rigid pocket; ensemble machinery redundant |
| Dasatinib >> Imatinib in D | Rotation direction wrong or pocket opens wrong way |
| All A~B~C~D | No discrimination signal; paper story fails |
| Scores diverge in D vs A/B/C | Domain rearrangement genuinely changes binding energy landscape |

### Pre-registered formula (decided before ABL1 results)
If fragmentation signal confirmed, penalise cluster count:
```
adjusted_score = best_score / log2(n_clusters + 2)
```
Scope: kinase inhibitor ligand sets only. Small rigid fragments (Ibuprofen, etc.) explicitly excluded.

## 11. Download results

In [ ]:
import zipfile, os
from IPython.display import FileLink, display

files = ["abl1_4way_results.csv", "abl1_4way_results.html"]
with zipfile.ZipFile("abl1_4way_experiment.zip", "w") as zf:
    for f in files:
        if os.path.exists(f):
            zf.write(f)
display(FileLink("abl1_4way_experiment.zip"))